This model attempts to predict whether a stock’s price will go up or down on the next trading day. Instead of looking at a single day in isolation, the model looks at a short sequence of recent days. For each prediction, it uses the previous 10 days of information to decide whether the price is more likely to increase or decrease tomorrow.

The model used here is called a GRU (Gated Recurrent Unit). A GRU is a type of neural network designed specifically to work with sequences, such as time-ordered data. It is well suited to problems where recent history may influence what happens next.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import RobustScaler
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve, confusion_matrix

## Configuration and hyperparameters
The sequence length defines how many past days the model will look at for each prediction. The maximum number of tickers limits how much data is loaded, which helps keep the experiment fast. Batch size and number of epochs control how the model is trained, and the device setting determines whether training runs on a GPU or CPU.

In [ ]:
DATA_DIR = Path("../data/raw/Stocks")
FOCUS_TICKERS = ["amzn","msft","nke","tsco"]

SEQ_LEN = 10
MAX_TICKERS = 400
BATCH = 256
EPOCHS = 5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## Loading the stock data
The script first gathers a list of stock files, up to the maximum number of tickers specified earlier. Each file is read into a table, the date column is converted into a proper date format, and a new column is added to identify which stock the data belongs to. All stock tables are then combined into one large dataset. Rows missing essential price or volume information are removed. Finally, the data is sorted by ticker and date so that each stock’s history is in correct chronological order.

In [ ]:
files = list(DATA_DIR.glob("*.txt"))[:MAX_TICKERS]
dfs = []
for f in files:
    try:
        df = pd.read_csv(f)
        df["Date"] = pd.to_datetime(df["Date"])
        df["Ticker"] = f.stem.replace(".us","")
        dfs.append(df)
    except:
        pass

df = pd.concat(dfs)
df = df.dropna(subset=["Open","High","Low","Close","Volume"])
df = df.sort_values(["Ticker","Date"])

## Creating the prediction target
For each stock, the closing price of the next day is compared to the closing price of the current day. If the price increases, the target value is set to 1. If it does not, the target value is set to 0. This operation is done separately for each stock so that price information from different stocks is never mixed.

In [ ]:
df["Target"] = (df.groupby("Ticker")["Close"].shift(-1) > df["Close"]).astype(int)

df["ret"] = (df["Close"] - df["Open"]) / df["Open"]
df["range"] = (df["High"] - df["Low"]) / df["Open"]
df["vol_chg"] = df.groupby("Ticker")["Volume"].pct_change()

## Creating input features
The daily return measures how much the price changed from the opening price to the closing price. The daily range measures how large the price movement was during the day. The volume change measures how trading activity changed compared to the previous day. These features are chosen because they summarize price behavior and trading activity in a compact numerical form that a neural network can process.

In [ ]:
FEATURES = ["ret","range","vol_chg"]

df.replace([np.inf,-np.inf],np.nan,inplace=True)
df = df.groupby("Ticker",group_keys=False).apply(lambda g: g.ffill().bfill())
df = df.dropna(subset=FEATURES+["Target"])

scaler = RobustScaler()
df[FEATURES] = scaler.fit_transform(df[FEATURES])

## Defining a custom dataset for sequences
For each stock, the data is broken into overlapping sequences of fixed length. Each sequence contains several consecutive days of feature values. The label for each sequence corresponds to what happens on the day immediately after the sequence ends. These sequences are stored as tensors, which are the data format PyTorch uses for neural networks. The dataset class also defines how many sequences exist and how to retrieve them one at a time. This step is essential because neural networks like GRUs expect sequence-shaped input rather than flat tables.

In [ ]:
class SeqDS(Dataset):
    def __init__(self, df):
        X,y=[],[]
        for _,g in df.groupby("Ticker"):
            arr=g[FEATURES].values
            tgt=g["Target"].values
            for i in range(SEQ_LEN,len(g)):
                X.append(arr[i-SEQ_LEN:i])
                y.append(tgt[i])
        self.X=torch.tensor(X,dtype=torch.float32)
        self.y=torch.tensor(y,dtype=torch.float32)
    def __len__(self): return len(self.X)
    def __getitem__(self,i): return self.X[i],self.y[i]

ds=SeqDS(df)
dl=DataLoader(ds,batch_size=BATCH,shuffle=True)

## Defining the GRU model
The model consists of a GRU layer followed by a fully connected output layer. The GRU processes each sequence and produces an internal representation that summarizes the information in the sequence. The final hidden state of the GRU is passed to the output layer, which produces a single value representing the model’s prediction. The model outputs a raw score rather than a probability. This design choice works together with the chosen loss function.

In [ ]:
class GRU(nn.Module):
    def __init__(self):
        super().__init__()
        self.gru=nn.GRU(len(FEATURES),32,batch_first=True)
        self.fc=nn.Linear(32,1)
    def forward(self,x):
        _,h=self.gru(x)
        return self.fc(h[-1]).squeeze()

model_name="GRU"
model=GRU().to(DEVICE)
opt=torch.optim.Adam(model.parameters(),lr=1e-3)
loss_fn=nn.BCEWithLogitsLoss()

## Training the model
Training runs for a fixed number of epochs. During each epoch, the model processes batches of sequences. For each batch, predictions are generated, the loss is calculated, and the model’s parameters are updated to reduce that loss. Gradients are cleared before each update, and backpropagation is used to determine how the model should change to improve its predictions.

In [ ]:
for e in range(EPOCHS):
    model.train()
    for xb,yb in dl:
        xb,yb=xb.to(DEVICE),yb.to(DEVICE)
        opt.zero_grad()
        loss=loss_fn(model(xb),yb)
        loss.backward()
        opt.step()

## Evaluating the model
After training, the model is switched to evaluation mode. The script passes all sequences through the model again to obtain predicted probabilities. These probabilities are stored along with the true labels. The probabilities are converted into binary predictions using a threshold of 0.5. This step produces the raw outputs needed for evaluation and plotting.

In [ ]:
model.eval()
all_probs,all_true=[],[]
with torch.no_grad():
    for xb,yb in dl:
        xb=xb.to(DEVICE)
        probs=torch.sigmoid(model(xb)).cpu().numpy()
        all_probs.append(probs)
        all_true.append(yb.numpy())

probs=np.concatenate(all_probs)
y_test=np.concatenate(all_true)
preds=(probs>=0.5).astype(int)

## Measuring model performance
Accuracy is calculated and compared to random guessing. A ROC curve is generated to assess how well the model separates upward and downward price movements. A confusion matrix shows the breakdown of correct and incorrect predictions. A probability histogram visualizes how confident the model is in its predictions. Together, these plots provide a broad understanding of how the model behaves.

In [ ]:
acc=accuracy_score(y_test,preds)
auc=roc_auc_score(y_test,probs)

# Accuracy
plt.figure(figsize=(4,3))
plt.bar(["Random (50%)",model_name],[0.5,acc])
plt.ylim(0.45,0.6)
plt.title("Accuracy vs Random")
plt.tight_layout()
plt.show()

# ROC
fpr,tpr,_=roc_curve(y_test,probs)
plt.figure(figsize=(4,4))
plt.plot(fpr,tpr,label=f"AUC={auc:.3f}")
plt.plot([0,1],[0,1],'k--')
plt.legend()
plt.title("ROC Curve")
plt.tight_layout()
plt.show()

# Confusion
cm=confusion_matrix(y_test,preds)
sns.heatmap(cm,annot=True,fmt="d",cmap="Blues")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()

# Probability
sns.histplot(probs,bins=50)
plt.axvline(0.5,color="k",linestyle="--")
plt.title("Predicted Probability Distribution")
plt.tight_layout()
plt.show()